In [ ]:
# Imports
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import squidpy as sq

from gpzoo.kernels import batched_MGGP_RBF
from gpzoo.model_utilities import (
    build_mggp_nsf_svgp,
    build_mggp_nsf_vnngp,
    init_mggp_inducing_points,
)


In [ ]:
# Paths and helpers
MODELS_DIR = Path('../../models/mggp_slideseq_benchmarks').resolve()
SUMMARY_GLOB = 'mggp_slideseq_benchmark_*.csv'
K_LIST = [5, 10, 25, 50, 100]
SVGP_INDUCING = 3000
SEED = 123

def _latest_summary_csv():
    files = sorted(MODELS_DIR.glob(SUMMARY_GLOB))
    return files[-1] if files else None

def _load_summary_df():
    csv_path = _latest_summary_csv()
    return pd.read_csv(csv_path) if csv_path else None

def _weights_path_svgp():
    return MODELS_DIR / 'slideseq_mggp_svgp.pth'

def _weights_path_k(k):
    return MODELS_DIR / f'slideseq_mggp_vnngp_k={k}.pth'

def _load_state_dict(model, weights_path, device):
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state)
    return model

def _loss_paths(weights_path):
    base = Path(weights_path)
    stem = base.stem
    csv_path = base.with_name(f"{stem}_losses.csv")
    npy_path = base.with_name(f"{stem}_losses.npy")
    return csv_path, npy_path

def _load_losses(weights_path):
    csv_path, npy_path = _loss_paths(weights_path)
    df = pd.read_csv(csv_path) if csv_path.exists() else None
    arr = np.load(npy_path) if npy_path.exists() else None
    return arr, df


In [ ]:
def plot_factors(factors, coords, title=None, size=3, s=4.0, vmin=None, vmax=None, cmap='turbo'):
    factors = np.asarray(factors)
    L = factors.shape[0]
    cols = min(4, L)
    rows = (L + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size), squeeze=False)
    for idx, ax in np.ndenumerate(axes):
        ax_idx = idx[0] * cols + idx[1]
        if ax_idx >= L:
            ax.axis('off')
            continue
        ax.scatter(coords[:, 0], coords[:, 1], c=factors[ax_idx], s=s, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_xticks([])
        ax.set_yticks([])
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    return fig


In [ ]:
# Load Slideseq data with benchmark preprocessing
import scanpy as sc
adata = sq.datasets.slideseqv2()
adata = adata.raw.to_adata()
adata.var['mt'] = adata.var_names.str.lower().str.startswith('mt-')
adata.var['MT'] = adata.var['mt']
adata = adata[adata.obs.pct_counts_mt < 20].copy()
sc.pp.filter_cells(adata, min_counts=100)
sc.pp.filter_genes(adata, min_cells=10)

gene_mask = ~adata.var['MT'].values
X_np = adata.obsm['spatial']
Y_matrix = adata[:, gene_mask].X
if hasattr(Y_matrix, 'toarray'):
    Y_matrix = Y_matrix.toarray()
Y_np = np.asarray(Y_matrix, dtype=np.float32).T
clusters = adata.obs['cluster'].astype('category')
groups_np = clusters.cat.codes.to_numpy()


In [ ]:
# Torch tensors
X = torch.tensor(X_np, dtype=torch.float32)
Y = torch.tensor(Y_np, dtype=torch.float32)
GROUPS = torch.tensor(groups_np, dtype=torch.long)
V = torch.ones(Y.shape[1], dtype=torch.float32)
device = torch.device('cpu')


In [ ]:
# Builders mirroring benchmark hyper-parameters
kernel = batched_MGGP_RBF(sigma=1.0, lengthscale=3.0, group_diff_param=10.0, n_groups=int(GROUPS.max().item()) + 1)

def make_svgp_model():
    target = min(SVGP_INDUCING, len(X))
    Z_init, groups_init = init_mggp_inducing_points(X, GROUPS, target, method='kmeans', seed=SEED)
    return build_mggp_nsf_svgp(
        X=X,
        groupsX=GROUPS,
        Y=Y,
        V=V,
        L=12,
        kernel=kernel,
        jitter=1e-5,
        inducing_points=Z_init,
        inducing_groups=groups_init,
        device=device,
        seed=SEED,
    )

def make_vnngp_model(k):
    return build_mggp_nsf_vnngp(
        X=X,
        groupsX=GROUPS,
        Y=Y,
        V=V,
        L=12,
        kernel=kernel,
        jitter=1e-5,
        K=k,
        lu_reference_points=X[::10],
        lu_reference_groups=GROUPS[::10],
        subset_step=10,
        device=device,
        seed=SEED,
        precompute_knn=True,
    )


In [ ]:
# Load SVGP model
svgp_model = make_svgp_model()
svgp_path = _weights_path_svgp()
svgp_model = _load_state_dict(svgp_model, svgp_path, device)
svgp_losses, svgp_losses_df = _load_losses(svgp_path)
svgp_model.eval()
with torch.no_grad():
    qF, _, _ = svgp_model.prior.forward(X, groupsX=GROUPS)
    svgp_mean = qF.mean.cpu().numpy()
    svgp_scale = qF.scale.cpu().numpy()


In [ ]:
# Load VNNGP models
vnngp_results = {}
for k in K_LIST:
    path = _weights_path_k(k)
    if not path.exists():
        continue
    model = make_vnngp_model(k)
    model = _load_state_dict(model, path, device)
    losses, losses_df = _load_losses(path)
    model.eval()
    with torch.no_grad():
        knn_idx = model.prior.calculate_knn(X)
        model.prior.knn_idx = knn_idx[:, :-1]
        qF, _, _ = model.prior.forward(X, groupsX=GROUPS)
        mean = qF.mean.cpu().numpy()
        scale = qF.scale.cpu().numpy()
    vnngp_results[k] = {
        'model': model,
        'losses': losses,
        'losses_df': losses_df,
        'mean': mean,
        'scale': scale,
    }


In [ ]:
summary_df = _load_summary_df()
summary_df


In [ ]:
# Loss curves
plt.figure(figsize=(10, 5))
if svgp_losses is not None:
    plt.plot(svgp_losses, label='SVGP')
for k, info in vnngp_results.items():
    if info['losses'] is not None:
        plt.plot(info['losses'], label=f'VNNGP K={k}')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Slideseq benchmark losses')
plt.legend()
plt.show()


In [ ]:
# Mean factors
vmax = np.percentile(np.exp(svgp_mean), 99)
plot_factors(np.exp(svgp_mean), X_np, title='MGGP SVGP mean factors', vmin=0, vmax=vmax, s=2.0)


In [ ]:
# Scale factors
vmax = np.percentile(svgp_scale, 99)
plot_factors(svgp_scale, X_np, title='MGGP SVGP scale factors', vmin=0, vmax=vmax, s=2.0)


In [ ]:
# VNNGP scales
for k, info in vnngp_results.items():
    vmax = np.percentile(info['scale'], 99)
    plot_factors(info['scale'], X_np, title=f'VNNGP K={k} scales', vmin=0, vmax=vmax, s=2.0)


In [ ]:
# Summary charts
if summary_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(summary_df['label'], summary_df['wall_time_sec'])
    axes[0].set_ylabel('seconds')
    axes[0].set_xticklabels(summary_df['label'], rotation=45, ha='right')
    axes[0].set_title('Wall-clock time')

    axes[1].bar(summary_df['label'], summary_df['final_loss'])
    axes[1].set_ylabel('final loss')
    axes[1].set_xticklabels(summary_df['label'], rotation=45, ha='right')
    axes[1].set_title('Final losses')
    plt.tight_layout()
    plt.show()
